In [47]:
import sys
import os
import requests
import pandas as pd
from io import BytesIO
from datetime import datetime, time

url = "https://1drv.ms/x/c/d6aca2526f83594b/IQAc8ap_zmWhR5goYf70_b69AU0CyhmpxFroY48j8pNaZac?download=1"

def get_data(url):
    response = requests.get(url)
    df = pd.read_excel(BytesIO(response.content), engine="openpyxl")
    return df

df = get_data(url)


print("program ran ... ")

program ran ... 


In [48]:
df_tow = df.copy()
print(df_tow.tail())

     Unnamed: 0 Unnamed: 1  2025-09-01 00:00:00     45901  \
1828        NaN        NaN  2026-04-07 00:00:00  10:04:00   
1829        NaN        NaN  2026-04-07 00:00:00  12:30:00   
1830        NaN        NaN  2026-04-07 00:00:00  16:00:00   
1831        NaN        NaN  2026-04-07 00:00:00  19:12:00   
1832        NaN        NaN  2026-04-07 00:00:00  21:12:00   

     *READINGS ARE FROM TOP OF CASING (TOC)* Unnamed: 5 Unnamed: 6 Unnamed: 7  \
1828                                   30.48      10.36      33.55      38.87   
1829                                    30.5      10.35      33.56      38.85   
1830                                    30.5      10.36      33.53      38.79   
1831                                   30.46      10.37      33.52      38.77   
1832                                   30.52      10.38      33.55      38.85   

     Unnamed: 8 Unnamed: 9 Unnamed: 10 Unnamed: 11 Unnamed: 12 Unnamed: 13  \
1828      31.95      40.32       42.15       36.55       29.64      

In [49]:
def shape_df(df):
    # cut the first 5 rows and first column 
    df = df.iloc[25:,2:].reset_index(drop=True)
    return df

df_tow = shape_df(df_tow)

print(df_tow.info())

# df_tow.to_excel('tow-test.xlsx', engine="openpyxl")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1808 entries, 0 to 1807
Data columns (total 15 columns):
 #   Column                                   Non-Null Count  Dtype 
---  ------                                   --------------  ----- 
 0   2025-09-01 00:00:00                      1808 non-null   object
 1   45901                                    1808 non-null   object
 2   *READINGS ARE FROM TOP OF CASING (TOC)*  1563 non-null   object
 3   Unnamed: 5                               1773 non-null   object
 4   Unnamed: 6                               1679 non-null   object
 5   Unnamed: 7                               1794 non-null   object
 6   Unnamed: 8                               1759 non-null   object
 7   Unnamed: 9                               1653 non-null   object
 8   Unnamed: 10                              1692 non-null   object
 9   Unnamed: 11                              1167 non-null   object
 10  Unnamed: 12                              1095 non-null   obj

In [50]:
print(len(df_tow.columns))
columns = ['date',	'time', 'TOW_1','TOW_2','TOW_3', 'TOW_4', 'TOW_5', 'TOW_6', 'TOW_7', 'TOW_8', 'TOW_15'	,'TOW_16','TOW_17','TOW_18', 'comments']
print(len(columns))
def trim_frame(df, new_col):
    df = df.iloc[:, :len(new_col)] 
    df.columns = new_col
    return df

def clean_col(df):
    # strip all columns with "Unnamed" string
    for col in df.columns:
        if "Unnamed" in str(col):
            df = df.drop(columns=[str(col)], errors='ignore')
    return df

df_tow = trim_frame(df_tow, columns)
df_tow = clean_col(df_tow)
print(df_tow.columns)

15
15
Index(['date', 'time', 'TOW_1', 'TOW_2', 'TOW_3', 'TOW_4', 'TOW_5', 'TOW_6',
       'TOW_7', 'TOW_8', 'TOW_15', 'TOW_16', 'TOW_17', 'TOW_18', 'comments'],
      dtype='object')


In [51]:
def fix_time(df):
    mask = df['time'].apply(lambda x: isinstance(x, datetime))
    df.loc[mask, 'time'] = df.loc[mask, 'time'].apply(lambda x: x.time())
    return df

def make_datetime_col(df):
    df['datetime'] = df.apply(
        lambda row: datetime.combine(row['date'].date(), row['time']),
        axis=1
    )
    return df
def fix_space(df):
    df.columns = df.columns.str.replace('.', '_', regex=False)
    df.columns = df.columns.str.replace(' ', '_', regex=False)  # replace spaces
    return df

def drop_DT(df):
    # strip all columns with "Unnamed" string
    df = df.drop(columns=['date', 'time'], errors='ignore')
    return df

df_tow = fix_time(df_tow)
df_tow = make_datetime_col(df_tow)
df_tow = fix_space(df_tow)
df_tow = drop_DT(df_tow)

print(df_tow)

      TOW_1  TOW_2  TOW_3  TOW_4  TOW_5  TOW_6  TOW_7  TOW_8 TOW_15 TOW_16  \
0       NaN    NaN    NaN   8.95    NaN    NaN    NaN    NaN    NaN   6.17   
1       NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN   7.45   
2       NaN    NaN    NaN   8.82    NaN    NaN    NaN    NaN    NaN   6.34   
3       NaN    NaN    NaN   8.82    NaN    NaN    NaN    NaN    NaN    6.3   
4       NaN    NaN    NaN   8.77    NaN    NaN    NaN    NaN    NaN   6.45   
...     ...    ...    ...    ...    ...    ...    ...    ...    ...    ...   
1803  30.48  10.36  33.55  38.87  31.95  40.32  42.15  36.55  29.64  41.17   
1804   30.5  10.35  33.56  38.85  31.95  40.33  42.17  36.58    NaN  41.09   
1805   30.5  10.36  33.53  38.79  31.94  40.19     42  36.47  29.62   41.5   
1806  30.46  10.37  33.52  38.77  31.95  40.15   41.9  36.39   29.6  40.75   
1807  30.52  10.38  33.55  38.85  31.99  40.22  41.88  36.42   29.6  40.97   

     TOW_17 TOW_18                                           co

In [52]:
print(df_tow.columns)

Index(['TOW_1', 'TOW_2', 'TOW_3', 'TOW_4', 'TOW_5', 'TOW_6', 'TOW_7', 'TOW_8',
       'TOW_15', 'TOW_16', 'TOW_17', 'TOW_18', 'comments', 'datetime'],
      dtype='object')


In [ ]:
def make_tidy(df):

    value_cols = [col for col in df.columns if 'TOW_' in col ]

    df_long = df.melt(
        id_vars='datetime',
        value_vars=value_cols,
        var_name='tow',
        value_name='value'
    )
    
    df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')
    df_long.columns.name = None
    return df_long

df_long = make_tidy(df_tow)
print(df_long.tail(10))




                 datetime     tow  value
21686 2026-04-06 21:12:00  TOW_18   51.4
21687 2026-04-07 00:00:00  TOW_18  51.42
21688 2026-04-07 02:00:00  TOW_18  51.45
21689 2026-04-07 05:10:00  TOW_18   51.4
21690 2026-04-07 06:55:00  TOW_18  51.45
21691 2026-04-07 10:04:00  TOW_18  51.47
21692 2026-04-07 12:30:00  TOW_18    NaN
21693 2026-04-07 16:00:00  TOW_18   51.3
21694 2026-04-07 19:12:00  TOW_18  51.27
21695 2026-04-07 21:12:00  TOW_18   51.3
